# 03 — Serum miRNA (GSE59856): p >> n, and a batch-effect warning

Phase 5. 571 serum samples, 2555 miRNAs. Performance numbers come from `reports/` (`python -m pancreasignal.train --config configs/mirna.yaml --task ...`). This notebook asks whether those numbers can be trusted.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import rankdata
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from pancreasignal.config import PROJECT_ROOT, load_config
from pancreasignal.data.geo import DETECTION_FLOOR, load_mirna, mirna_columns, mirna_names, select_mirna_task
from pancreasignal.explain import predict_other_groups, selection_stability
from pancreasignal.features import build_mirna_features
from pancreasignal.models import build_elasticnet
from pancreasignal.reporting import MIRNA_TASKS, MIRNA_VARIANTS, get_run, load_runs

sns.set_theme(style="whitegrid", context="notebook")
FIG_DIR = PROJECT_ROOT / "docs" / "figures"
CFG = load_config("configs/mirna.yaml")
PARAMS = CFG["model"]["params"]["elasticnet"]
CV = StratifiedKFold(5, shuffle=True, random_state=42)

df = load_mirna()
cols = mirna_columns(df)
names = mirna_names()
print(df.shape, "|", len(cols), "miRNAs")

## 1. Who is in the data?
Groups, the sample-title prefix (a hint of how samples were collected), age, sex, stage.

In [ ]:
display(pd.crosstab(df.diagnosis, df.title_prefix))
display(df.groupby("diagnosis").agg(n=("age", "size"), median_age=("age", "median"),
                                    male=("sex", lambda s: round((s == "M").mean(), 2))))
df[df.diagnosis == "pdac"].stage.value_counts()

**What this shows.**
- 8 groups. The title prefix matches the diagnosis almost everywhere (P = pancreatic, B = biliary, S/E = healthy, I = benign, ...), except a few swaps between P and B (pancreatic, biliary, liver): those collections were probably handled together and relabelled after the final diagnosis.
- Healthy controls are 91% male (pancreatic cancer 64%) and about 5 years younger, so an age + sex baseline is needed.
- Pancreatic cancer is mostly late stage: 81 of 100 are stage III–IV, only 19 are stage I–II (1 is stage I).

## 2. Quality: how many miRNAs are detected?
GEO replaced every undetected value with one floor value (2.612). Left: per sample. Right: per miRNA.

In [ ]:
df["n_detected"] = (df[cols] > DETECTION_FLOOR).sum(axis=1)
detect_rate = (df[cols] > DETECTION_FLOOR).mean()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
order = df.groupby("diagnosis").n_detected.median().sort_values().index
sns.boxplot(data=df, x="diagnosis", y="n_detected", order=order, ax=axes[0], color="#9BB8D3")
axes[0].set(title="Detected miRNAs per sample", xlabel="", ylabel="miRNAs above the floor")
axes[0].tick_params(axis="x", rotation=30)
axes[1].hist(detect_rate, bins=40, color="#9BB8D3")
axes[1].axvline(PARAMS["min_detection_rate"], color="#C44E52", ls="--", label="filter: kept if >= 50% (per training fold)")
axes[1].set(title="Share of samples in which each miRNA is detected", xlabel="detection rate", ylabel="miRNAs")
axes[1].legend()
plt.tight_layout(); plt.show()

print(f"{(detect_rate >= 0.5).sum()} of {len(cols)} miRNAs are detected in >= 50% of samples")
print("Arrays with > 2000 miRNAs detected (suspicious: almost everything above background):")
df.loc[df.n_detected > 2000, ["sample_id", "title", "diagnosis", "n_detected"]]

**What this shows.**
- Most miRNAs are either almost never or almost always detected (the two peaks); 883 of 2555 pass a 50% detection rate. The filter is applied per training fold, inside the Pipeline.
- 10 arrays report almost all 2555 miRNAs as detected, which looks like a failed background correction. None of them are pancreatic cancer or controls (2 are biliary). They are kept here; a real study would remove them in quality control.

## 3. p >> n: the leakage trap, shown with random labels
Shuffle the labels, so there is **no** real signal. Then: (a) pick the 20 "best" miRNAs using all samples, and only then cross-validate; (b) do the same selection inside each training fold; (c) run our real elastic-net Pipeline. Honest methods must give AUROC around 0.5.

In [ ]:
rows, y = select_mirna_task(df, "pdac_vs_control")
X = rows[cols].fillna(DETECTION_FLOOR).to_numpy()   # simple fill: this demo is about the selection step

results = []
for seed in range(5):
    y_random = np.random.default_rng(seed).permutation(y.to_numpy())

    # (a) WRONG: choose miRNAs on all samples (the test folds included), then cross-validate
    X_top = SelectKBest(f_classif, k=20).fit_transform(X, y_random)
    p_leak = cross_val_predict(LogisticRegression(max_iter=2000), X_top, y_random, cv=CV, method="predict_proba")[:, 1]

    # (b) RIGHT: the same selection, but inside the Pipeline -> refitted on each training fold only
    pipe = Pipeline([("select", SelectKBest(f_classif, k=20)), ("scale", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=2000))])
    p_ok = cross_val_predict(pipe, X, y_random, cv=CV, method="predict_proba")[:, 1]
    results.append({"seed": seed, "(a) select on all data": roc_auc_score(y_random, p_leak),
                    "(b) select inside CV": roc_auc_score(y_random, p_ok)})

# (c) our real Pipeline (imputer + detection filter + nested elastic-net), one shuffle
y_random = np.random.default_rng(0).permutation(y.to_numpy())
p_ours = cross_val_predict(build_elasticnet(PARAMS), rows[cols], y_random, cv=CV, method="predict_proba")[:, 1]
leak = pd.DataFrame(results).set_index("seed").round(3)
display(leak)
print(f"(c) our elastic-net Pipeline on shuffled labels: AUROC {roc_auc_score(y_random, p_ours):.3f}")

**What this shows.**
- With shuffled labels there is nothing to find, yet choosing the 20 "best" miRNAs on all samples first gives AUROC 0.72–0.78 every time. With 2555 candidates, some always fit the noise of these exact samples, test folds included.
- The same selection inside the CV gives about 0.5, and our elastic-net Pipeline 0.54. So the pipeline does not leak, and the near-perfect scores in section 4 are not a code bug.
- The core p >> n lesson: **every** data-driven choice (feature selection, filtering, tuning) must be refitted inside each training fold.

## 4. Results from `reports/`
Pooled out-of-fold AUROC, the age + sex baseline, and how many miRNAs each outer-fold model kept (non-zero coefficients).

In [ ]:
runs = load_runs()
runs = runs[runs.dataset == "mirna"]
summary = pd.DataFrame({
    MIRNA_TASKS[t]: {MIRNA_VARIANTS[v]: get_run(runs, t, v)["results"]["model_all"]["auroc"] for v in MIRNA_VARIANTS}
    for t in MIRNA_TASKS
}).round(3)
display(summary)
pd.DataFrame({MIRNA_TASKS[t]: {f: d.get("n_nonzero_coefs") for f, d in get_run(runs, t, "mirna")["results"]["per_fold"].items()}
              for t in MIRNA_TASKS})

**What this shows.**
- PDAC vs control and PDAC vs benign: almost perfect, far above age + sex. PDAC vs biliary: chance, and the inner CV chose the empty model (no miRNA at all) in 4 of 5 folds: it found nothing that generalises.
- For PDAC vs control each fold keeps roughly 460–650 miRNAs: with a modest L1 share, elastic-net keeps large groups of correlated miRNAs.

## 5. Is it cancer, or is it "not a healthy control"?
**(a)** For every pair of groups: the best AUROC any **single** miRNA reaches. With ~50+ samples per group, chance alone gives roughly 0.65–0.75 for the best of 2555 miRNAs; values near 1.0 are a real, very strong difference.

In [ ]:
def best_single_auroc(a, b):
    """Best univariate AUROC (folded to 0.5-1) over all miRNAs for group a vs group b. Vectorised Mann-Whitney."""
    s = df.diagnosis.isin([a, b])
    vals = df.loc[s, cols].fillna(DETECTION_FLOOR).to_numpy()
    is_a = (df.loc[s, "diagnosis"] == a).to_numpy()
    ranks = rankdata(vals, axis=0)                      # ties (the floor) get average ranks
    n_a, n_b = is_a.sum(), (~is_a).sum()
    auc = (ranks[is_a].sum(axis=0) - n_a * (n_a + 1) / 2) / (n_a * n_b)
    return np.maximum(auc, 1 - auc).max()

groups = ["control", "benign", "pdac", "biliary", "liver", "stomach", "esophagus", "colon"]
matrix = pd.DataFrame(np.nan, index=groups, columns=groups)
for i, a in enumerate(groups):
    for b in groups[i + 1:]:
        matrix.loc[a, b] = matrix.loc[b, a] = best_single_auroc(a, b)

fig, ax = plt.subplots(figsize=(7.5, 6))
sns.heatmap(matrix, annot=True, fmt=".2f", cmap="rocket_r", vmin=0.6, vmax=1.0, square=True,
            cbar_kws={"label": "best single-miRNA AUROC"}, ax=ax)
ax.set_title("How well can ONE miRNA separate two groups?")
plt.tight_layout()
fig.savefig(FIG_DIR / "mirna_group_separation.png", dpi=150, bbox_inches="tight")
plt.show()

**(b)** Fit the PDAC-vs-control model on all PDAC and control samples, then score the groups it never saw. A pancreatic-cancer test should **not** call benign disease or colon cancer "PDAC".

In [ ]:
cfg = load_config("configs/mirna.yaml")
cfg["task"] = "pdac_vs_control"
predict_other_groups(cfg).round(2)

**(c)** The healthy controls come in two title batches, "S" (86) and "E" (64). Can the same Pipeline tell the two batches of healthy people apart?

In [ ]:
ctrl = df[df.diagnosis == "control"].reset_index(drop=True)
y_batch = (ctrl.title_prefix == "S").astype(int)
p_batch = cross_val_predict(build_elasticnet(PARAMS), ctrl[cols], y_batch, cv=CV, method="predict_proba")[:, 1]
print(f"Healthy controls, batch S vs batch E: out-of-fold AUROC {roc_auc_score(y_batch, p_batch):.3f}")
display(ctrl.groupby("title_prefix").agg(n=("age", "size"), median_age=("age", "median"),
                                         male=("sex", lambda s: round((s == "M").mean(), 2))))

**What this shows.**
- **(a)** One single miRNA separates healthy controls from every other group (0.98–1.00). Most pairs of disease groups are also separated by one miRNA; the clear exception is pancreatic vs biliary cancer (0.68), the two groups that seem to have been collected together.
- **(b)** The PDAC-vs-control model calls most samples it never saw "PDAC": all biliary cancers and most esophagus, benign, liver, stomach and colon samples. It has learned "not a healthy control", not pancreatic cancer.
- **(c)** Two batches of **healthy** people (same median age, both mostly male) are separated with AUROC 0.998. No biology should do that; it is a collection or processing batch effect.
- Conclusion: in this dataset, group and collection batch are mixed together. The perfect AUROCs cannot be credited to cancer, and the data cannot tell how much real cancer signal is left.

## 6. Which miRNAs? Selection stability
For PDAC vs control: each outer-fold model keeps its own set of miRNAs. How many are kept in all 5 folds?

In [ ]:
cfg = load_config("configs/mirna.yaml")
cfg["task"] = "pdac_vs_control"
stability = selection_stability(cfg)
stability.insert(0, "name", [names.get(i, i) for i in stability.index])
print(f"kept at least once: {len(stability)}  |  kept in all 5 folds: {(stability.n_folds == 5).sum()}")
stability.head(15).round(3)

**What this shows.**
- 825 miRNAs are kept at least once and 342 in all 5 folds, so "the model's miRNAs" is not one short list. Report the stable ones and how often they were kept.
- hsa-miR-125a-3p is kept in every fold and is also the best single separator of pancreatic cancer from controls. Compare the stable list with the panel published with GSE59856 before claiming agreement; given section 5, agreement would still not prove a cancer signal.
- Negative coefficient = lower in pancreatic cancer than in controls.

## Summary

- **p >> n done right:** filtering, scaling and the penalty choice all happen inside each training fold (nested CV), and shuffled labels give ~0.5. Selecting on all data first "finds" signal in pure noise (~0.75).
- **But this data cannot answer the clinical question.** Near-perfect separation appears wherever groups were collected separately, even between two batches of healthy people, and disappears for the two cancers collected together.
- **The urine model is the more trustworthy result**, even with lower numbers: it was tested on hospitals it never saw, against CA19-9 and an age baseline.
- **Interview line:** "My miRNA model scored AUROC 1.0, and I showed why that number should not be trusted."